In [39]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('./data/cleaned_data.csv')

C:\Users\vecto\AppData\Local\Temp\ipykernel_26252\1827708996.py:6: DtypeWarning: Columns (0: verification_status_joint, 1: sec_app_earliest_cr_line) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/cleaned_data.csv')


In [40]:
print(df['issue_d'].min(), df['issue_d'].max())

2007-06-01 2018-12-01


Set up train-test split:

In [41]:
df['issue_d'] = pd.to_datetime(df['issue_d'])
categorical_cols = ['purpose', 'home_ownership', 'verification_status']
for col in categorical_cols:
    df[col] = df[col].astype('category')

In [42]:
split_date = pd.Timestamp('2017-01-01')
train = df[df['issue_d'] < split_date]
test = df[df['issue_d'] >= split_date]
print(train.shape, test.shape)

(1321847, 102) (938821, 102)


In [43]:
features = [
    'loan_amnt', 'term', 'purpose', 'emp_length', 'home_ownership',
    'annual_inc', 'verification_status', 'dti', 'delinq_2yrs',
    'fico_range_low', 'fico_range_high', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'mort_acc', 'credit_history_months'
]

X_train = train.drop(columns=['int_rate', 'issue_d'])[features]
y_train = train['int_rate']
X_test = test.drop(columns=['int_rate', 'issue_d'])[features]
y_test = test['int_rate']

Fit a LightGBM model:

In [44]:
model = lgb.LGBMRegressor(random_state=42)
model.fit(X_train, y_train, categorical_feature=categorical_cols)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 1321847, number of used features: 19
[LightGBM] [Info] Start training from score 13.178320
